In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
data_path = Path("../../data/gold_layer/herd_mentality_events_gold_data_v1.csv").resolve()
preprocessed_path = Path("../../data/gold_layer/herd_mentality_events_gold_data_v2.csv").resolve()

In [3]:
df = pd.read_csv(data_path)
df

,Year,Event Name,Continent,Event Description,Country,Decade,Event Type,Trigger,Magnitude(M),Spread(S),Intensity(I),Duration(D),Outcome/Impact(R),Measure_label,Spread_label,Intensity_label,Duration_label,Outcome_label
0,1925,South African Rand mine strike,Africa,"In 1925, thousands of African gold‑mine worker...",South Africa,1920's,Social,Repression,8.0,4.0,7.0,5.0,8.0,Very High,Regional,High,Moderate,Major
1,1926,Birth of Pan-African Congress,Africa,"In 1926, African intellectuals and diaspora le...",United Kingdom,1920's,Social,Policy,6.0,9.0,5.0,3.0,9.0,Moderate-High,Continental,Moderate,Short,Transformational
2,1927,Foundation of ANC Youth League (SA),Africa,"In 1927, young activists in South Africa found...",South Africa,1920's,Social,Repression,4.0,6.0,6.0,4.0,7.0,Moderate-Low,National,Moderate-High,Moderate-Short,Significant
3,1928,South African Black political conference,Africa,"In 1928, black political leaders from across S...",South Africa,1920's,Social,Repression,5.0,6.0,3.0,3.0,7.0,Moderate,National,Low,Short,Significant
4,1930,Mass migration for economic reasons in Sahel,Africa,In 1930 a severe drought combined with the glo...,Mali,1930's,Social,Policy,7.0,8.0,8.0,6.0,7.0,High,Multi-National,Very High,Long,Significant
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
620,2021,Papua volcano eruption response,Australia/Oceania,"In December 2021, Mount Manam in Papua New Gui...",Papua New Guinea,2020's,Social,Fear,6.0,6.0,7.0,5.0,7.0,Moderate-High,National,High,Moderate,Significant
621,2022,Fiji cyclone mass evacuations,Australia/Oceania,"In February 2022, Fiji’s Meteorological Servic...",Fiji,2020's,Social,Fear,5.0,6.0,7.0,5.0,5.0,Moderate,National,High,Moderate,Moderate
622,2023,Indigenous Voice to Parliament activism (Austr...,Australia/Oceania,"Throughout 2023, Australia saw a nationwide su...",Australia,2020's,Social,Policy,6.0,6.0,6.0,8.0,4.0,Moderate-High,National,Moderate-High,Very Long,Moderate-Low
623,2024,Pacific Islands anti-mining protests,Australia/Oceania,"In 2024, island communities across the Pacific...",Fiji,2020's,Social,Fear,6.0,8.0,7.0,5.0,7.0,Moderate-High,Multi-National,High,Moderate,Significant


In [4]:
# Function to classify dimension values into Low/Medium/High segments
def classify_segment(value):
    """
    Classify a dimension value into Low, Medium, or High based on ranges:
    - Low: 1.0-3.0 (inclusive)
    - Medium: 4.0-6.0 (inclusive)
    - High: 7.0-10.0 (inclusive)
    
    Handles float values like 1.0, 2.0, 3.0, etc.
    """
    if pd.isna(value):
        return None
    
    # Convert to float to handle both int and float inputs
    value = float(value)
    
    # Classify based on ranges
    if 1.0 <= value <= 3.0:
        return "Low"
    elif 4.0 <= value <= 6.0:
        return "Medium"
    elif 7.0 <= value <= 10.0:
        return "High"
    else:
        return None

def get_mode(value):
    """
    If the Spread (S) Value is in between 1.0 and 3.0, then the mode is Co-present else it is difffusive
    """
    if 1.0 <= value <= 3.0:
        return "Co-present"
    else:
        return "Diffusive"

def assign_segment(row):
    """
    Assign segment (S1-S5) based on dimension values.
    Rules are checked in order of specificity (most specific first).
    
    S1 — Aligned Expansion (Value-Creating leaning)
    S2 — Emotion-Driven (Unstable / Transitional)
    S3 — Volatile Expansion (Looks big, doesn't last)
    S4 — Persistent Friction (Value-Destructive leaning)
    S5 — Low-Energy Diffusion (Low-signal)
    """
    # Extract dimension values
    M = row.get('Magnitude(M)', np.nan)
    S = row.get('Spread(S)', np.nan)
    I = row.get('Intensity(I)', np.nan)
    D = row.get('Duration(D)', np.nan)
    mode = row.get('Mode', np.nan)
    
    # Check for missing values
    if pd.isna(M) or pd.isna(S) or pd.isna(I) or pd.isna(D) or pd.isna(mode):
        return None
    
    # Convert to float
    M, S, I, D = float(M), float(S), float(I), float(D)
    
    # Helper function to check segment level
    def is_low(val): return 1.0 <= val <= 3.0
    def is_medium(val): return 4.0 <= val <= 6.0
    def is_high(val): return 7.0 <= val <= 10.0
    def is_medium_or_above(val): return val >= 4.0
    
    # Apply rules in order of specificity (most specific first)
    
    # S5 — Low-Energy Diffusion (Low-signal)
    # IF (I == Low)
    if is_low(I) and mode == "Diffusive":
        return "S5 - Low Energy Diffusion"
    
    # S4 — Persistent Friction (Value-Destructive leaning)
    # IF (I == High) AND (D == High)
    if is_high(I) and is_high(D):
        return "S4 - Persistent Friction"
    
    # S2 — Emotion-Driven (Unstable / Transitional)
    # IF (I == High) AND (D < High)
    if is_high(I) and not is_high(D):
        return "S2 - Emotion-Driven"
    
    # S3 — Volatile Expansion (Looks big, doesn't last)
    # IF (M >= Medium AND S >= Medium) AND (D == Low)
    if is_medium_or_above(M) and is_medium_or_above(S) and is_low(D):
        return "S3 - Volatile Expansion"
    
    # S1 — Aligned Expansion (Value-Creating leaning)
    # IF (M >= Medium AND S >= Medium) AND (D >= Medium) AND (I == Medium)
    if is_medium_or_above(M) and is_medium_or_above(S) and is_medium_or_above(D) and is_medium(I):
        return "S1 - Aligned Expansion"
    
    # If no segment matches, return None (unclassified)
    return "None"

In [5]:
df['Mode'] = df['Spread(S)'].apply(get_mode)
df

,Year,Event Name,Continent,Event Description,Country,Decade,Event Type,Trigger,Magnitude(M),Spread(S),Intensity(I),Duration(D),Outcome/Impact(R),Measure_label,Spread_label,Intensity_label,Duration_label,Outcome_label,Mode
0,1925,South African Rand mine strike,Africa,"In 1925, thousands of African gold‑mine worker...",South Africa,1920's,Social,Repression,8.0,4.0,7.0,5.0,8.0,Very High,Regional,High,Moderate,Major,Diffusive
1,1926,Birth of Pan-African Congress,Africa,"In 1926, African intellectuals and diaspora le...",United Kingdom,1920's,Social,Policy,6.0,9.0,5.0,3.0,9.0,Moderate-High,Continental,Moderate,Short,Transformational,Diffusive
2,1927,Foundation of ANC Youth League (SA),Africa,"In 1927, young activists in South Africa found...",South Africa,1920's,Social,Repression,4.0,6.0,6.0,4.0,7.0,Moderate-Low,National,Moderate-High,Moderate-Short,Significant,Diffusive
3,1928,South African Black political conference,Africa,"In 1928, black political leaders from across S...",South Africa,1920's,Social,Repression,5.0,6.0,3.0,3.0,7.0,Moderate,National,Low,Short,Significant,Diffusive
4,1930,Mass migration for economic reasons in Sahel,Africa,In 1930 a severe drought combined with the glo...,Mali,1930's,Social,Policy,7.0,8.0,8.0,6.0,7.0,High,Multi-National,Very High,Long,Significant,Diffusive
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
620,2021,Papua volcano eruption response,Australia/Oceania,"In December 2021, Mount Manam in Papua New Gui...",Papua New Guinea,2020's,Social,Fear,6.0,6.0,7.0,5.0,7.0,Moderate-High,National,High,Moderate,Significant,Diffusive
621,2022,Fiji cyclone mass evacuations,Australia/Oceania,"In February 2022, Fiji’s Meteorological Servic...",Fiji,2020's,Social,Fear,5.0,6.0,7.0,5.0,5.0,Moderate,National,High,Moderate,Moderate,Diffusive
622,2023,Indigenous Voice to Parliament activism (Austr...,Australia/Oceania,"Throughout 2023, Australia saw a nationwide su...",Australia,2020's,Social,Policy,6.0,6.0,6.0,8.0,4.0,Moderate-High,National,Moderate-High,Very Long,Moderate-Low,Diffusive
623,2024,Pacific Islands anti-mining protests,Australia/Oceania,"In 2024, island communities across the Pacific...",Fiji,2020's,Social,Fear,6.0,8.0,7.0,5.0,7.0,Moderate-High,Multi-National,High,Moderate,Significant,Diffusive


In [6]:
# Apply segment assignment
df['Segment'] = df.apply(assign_segment, axis=1)

# Display the updated dataframe with segment column
print(f"Total events: {len(df)}")
print(f"Events with segments: {df['Segment'].notna().sum()}")
print(f"\nSegment distribution:")
print(df['Segment'].value_counts(dropna=False))
print(f"\nFirst few rows with Segment column:")
df[['Event Name', 'Year', 'Magnitude(M)', 'Spread(S)', 'Intensity(I)', 'Duration(D)', 'Segment']].head(10)

Total events: 625
Events with segments: 625

Segment distribution:
Segment
S2 - Emotion-Driven          264
S4 - Persistent Friction     149
S1 - Aligned Expansion       148
S3 - Volatile Expansion       49
None                          11
S5 - Low Energy Diffusion      4
Name: count, dtype: int64

First few rows with Segment column:


,Event Name,Year,Magnitude(M),Spread(S),Intensity(I),Duration(D),Segment
0,South African Rand mine strike,1925,8.0,4.0,7.0,5.0,S2 - Emotion-Driven
1,Birth of Pan-African Congress,1926,6.0,9.0,5.0,3.0,S3 - Volatile Expansion
2,Foundation of ANC Youth League (SA),1927,4.0,6.0,6.0,4.0,S1 - Aligned Expansion
3,South African Black political conference,1928,5.0,6.0,3.0,3.0,S5 - Low Energy Diffusion
4,Mass migration for economic reasons in Sahel,1930,7.0,8.0,8.0,6.0,S2 - Emotion-Driven
5,Libya: Resistance against Italian colonization,1931,7.0,6.0,8.0,6.0,S2 - Emotion-Driven
6,Egypt: Mass protests for independence,1932,7.0,6.0,7.0,5.0,S2 - Emotion-Driven
7,"Anticolonial riots, Kenya",1933,6.0,4.0,7.0,3.0,S2 - Emotion-Driven
8,Moroccan urban protests,1934,6.0,6.0,7.0,3.0,S2 - Emotion-Driven
9,Italian invasion of Ethiopia (mobilizations),1935,7.0,7.0,8.0,8.0,S4 - Persistent Friction


In [7]:
# Display full segment analysis
print("=" * 80)
print("SEGMENT ANALYSIS")
print("=" * 80)

# Segment distribution with percentages
segment_counts = df['Segment'].value_counts(dropna=False)
segment_pct = df['Segment'].value_counts(normalize=True, dropna=False) * 100

print("\nSegment Distribution:")
for seg in ['S1 - Aligned Expansion', 'S2 - Emotion-Driven', 'S3 - Volatile Expansion', 'S4 - Persistent Friction', 'S5 - Low-Energy Diffusion', 'None']:
    count = segment_counts.get(seg, 0)
    pct = segment_pct.get(seg, 0)
    print(f"  {seg}: {count:3d} events ({pct:5.2f}%)")

SEGMENT ANALYSIS

Segment Distribution:
  S1 - Aligned Expansion: 148 events (23.68%)
  S2 - Emotion-Driven: 264 events (42.24%)
  S3 - Volatile Expansion:  49 events ( 7.84%)
  S4 - Persistent Friction: 149 events (23.84%)
  S5 - Low-Energy Diffusion:   0 events ( 0.00%)
  None:  11 events ( 1.76%)


In [8]:
# Show sample events for each segment
print("\n" + "=" * 80)
print("SAMPLE EVENTS BY SEGMENT")
print("=" * 80)

segment_names = {
    'S1 - Aligned Expansion': 'Aligned Expansion (Value-Creating leaning)',
    'S2 - Emotion-Driven': 'Emotion-Driven (Unstable / Transitional)',
    'S3 - Volatile Expansion': 'Volatile Expansion (Looks big, doesn\'t last)',
    'S4 - Persistent Friction': 'Persistent Friction (Value-Destructive leaning)',
    'S5 - Low-Energy Diffusion': 'Low-Energy Diffusion (Low-signal)',
    'None': 'Unclassified'
}

for seg in ['S1 - Aligned Expansion', 'S2 - Emotion-Driven', 'S3 - Volatile Expansion', 'S4 - Persistent Friction', 'S5 - Low-Energy Diffusion', 'None']:
    seg_events = df[df['Segment'] == seg]
    if len(seg_events) > 0:
        print(f"\n{seg} — {segment_names[seg]}:")
        sample = seg_events[['Event Name', 'Year', 'Magnitude(M)', 'Spread(S)', 'Intensity(I)', 'Duration(D)', 'Segment']].head(3)
        print(sample.to_string(index=False))
        print()


SAMPLE EVENTS BY SEGMENT

S1 - Aligned Expansion — Aligned Expansion (Value-Creating leaning):
                                 Event Name  Year  Magnitude(M)  Spread(S)  Intensity(I)  Duration(D)                Segment
        Foundation of ANC Youth League (SA)  1927           4.0        6.0           6.0          4.0 S1 - Aligned Expansion
    Mass conscription in French West Africa  1940           6.0        4.0           6.0          9.0 S1 - Aligned Expansion
Staged labor actions in Nigerian oil fields  1944           5.0        6.0           6.0          4.0 S1 - Aligned Expansion


S2 - Emotion-Driven — Emotion-Driven (Unstable / Transitional):
                                    Event Name  Year  Magnitude(M)  Spread(S)  Intensity(I)  Duration(D)             Segment
                South African Rand mine strike  1925           8.0        4.0           7.0          5.0 S2 - Emotion-Driven
  Mass migration for economic reasons in Sahel  1930           7.0        8.0           

In [9]:
display(df)

,Year,Event Name,Continent,Event Description,Country,Decade,Event Type,Trigger,Magnitude(M),Spread(S),Intensity(I),Duration(D),Outcome/Impact(R),Measure_label,Spread_label,Intensity_label,Duration_label,Outcome_label,Mode,Segment
0,1925,South African Rand mine strike,Africa,"In 1925, thousands of African gold‑mine worker...",South Africa,1920's,Social,Repression,8.0,4.0,7.0,5.0,8.0,Very High,Regional,High,Moderate,Major,Diffusive,S2 - Emotion-Driven
1,1926,Birth of Pan-African Congress,Africa,"In 1926, African intellectuals and diaspora le...",United Kingdom,1920's,Social,Policy,6.0,9.0,5.0,3.0,9.0,Moderate-High,Continental,Moderate,Short,Transformational,Diffusive,S3 - Volatile Expansion
2,1927,Foundation of ANC Youth League (SA),Africa,"In 1927, young activists in South Africa found...",South Africa,1920's,Social,Repression,4.0,6.0,6.0,4.0,7.0,Moderate-Low,National,Moderate-High,Moderate-Short,Significant,Diffusive,S1 - Aligned Expansion
3,1928,South African Black political conference,Africa,"In 1928, black political leaders from across S...",South Africa,1920's,Social,Repression,5.0,6.0,3.0,3.0,7.0,Moderate,National,Low,Short,Significant,Diffusive,S5 - Low Energy Diffusion
4,1930,Mass migration for economic reasons in Sahel,Africa,In 1930 a severe drought combined with the glo...,Mali,1930's,Social,Policy,7.0,8.0,8.0,6.0,7.0,High,Multi-National,Very High,Long,Significant,Diffusive,S2 - Emotion-Driven
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
620,2021,Papua volcano eruption response,Australia/Oceania,"In December 2021, Mount Manam in Papua New Gui...",Papua New Guinea,2020's,Social,Fear,6.0,6.0,7.0,5.0,7.0,Moderate-High,National,High,Moderate,Significant,Diffusive,S2 - Emotion-Driven
621,2022,Fiji cyclone mass evacuations,Australia/Oceania,"In February 2022, Fiji’s Meteorological Servic...",Fiji,2020's,Social,Fear,5.0,6.0,7.0,5.0,5.0,Moderate,National,High,Moderate,Moderate,Diffusive,S2 - Emotion-Driven
622,2023,Indigenous Voice to Parliament activism (Austr...,Australia/Oceania,"Throughout 2023, Australia saw a nationwide su...",Australia,2020's,Social,Policy,6.0,6.0,6.0,8.0,4.0,Moderate-High,National,Moderate-High,Very Long,Moderate-Low,Diffusive,S1 - Aligned Expansion
623,2024,Pacific Islands anti-mining protests,Australia/Oceania,"In 2024, island communities across the Pacific...",Fiji,2020's,Social,Fear,6.0,8.0,7.0,5.0,7.0,Moderate-High,Multi-National,High,Moderate,Significant,Diffusive,S2 - Emotion-Driven


In [10]:
# Save the dataframe with Segment column to the preprocessed path
df.to_csv(preprocessed_path, index=False)
print(f"✓ Data saved to: {preprocessed_path}")
print(f"✓ Total columns: {len(df.columns)}")
print(f"✓ Total rows: {len(df)}")
print(f"\nColumns in saved file:")
print(df.columns.tolist())

✓ Data saved to: C:\RohitDir\Martian Data\data\gold_layer\herd_mentality_events_gold_data_v2.csv
✓ Total columns: 20
✓ Total rows: 625

Columns in saved file:
['Year', 'Event Name', 'Continent', 'Event Description', 'Country', 'Decade', 'Event Type', 'Trigger', 'Magnitude(M)', 'Spread(S)', 'Intensity(I)', 'Duration(D)', 'Outcome/Impact(R)', 'Measure_label', 'Spread_label', 'Intensity_label', 'Duration_label', 'Outcome_label', 'Mode', 'Segment']
